In [0]:
%sql
SELECT current_catalog(), current_schema();


In [0]:
%skip
source_path = "/Volumes/workspace/default/lab_data/users-historical/"

df = spark.read.format("csv").load(source_path)
df.show()

In [0]:
%skip
(df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(f"{catalog}.{schema}.historical_users_bronze_python")
 )

In [0]:
%skip
users_bronze_table = spark.table(f"{catalog}.{schema}.historical_users_bronze_python")
users_bronze_table.display()


In [0]:
%skip
df.printSchema()

In [0]:
%skip
%sql

CREATE OR REPLACE TABLE historical_users_bronze_ctas_rf
SELECT * FROM read_files(
    "/Volumes/workspace/default/lab_data/users-historical/",
    format => "csv",
    header => false,
    schema => "user_id STRING,
    user_first_touch_timestamp BIGINT,
    email STRING"
    );

SELECT * FROM historical_users_bronze_ctas_rf;

In [0]:
# from pyspark.sql.types import StructType, StructField, StringType,IntegerType,TimestampType,bigIntType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, LongType
schema = StructType([
  StructField("userId", StringType(), True),
  StructField("user_first_touch_timestamp", LongType(), True),
  StructField("user_email", StringType(), True)
])

In [0]:
%sql
CREATE OR REPLACE TABLE historical_users_bronze_ctas_rf
(
    userId STRING,
    user_first_touch_timestamp BIGINT
) USING delta;


COPY INTO historical_users_bronze_ctas_rf
FROM (SELECT _c0 as userId,cast(_c1 AS bigint) as user_first_touch_timestamp, _c2 as user_email
FROM "/Volumes/workspace/default/lab_data/users-historical/")
FILEFORMAT = CSV
FORMAT_OPTIONS ("header" = "false")
COPY_OPTIONS("mergeSchema" = "true");
    
SELECT * FROM historical_users_bronze_ctas_rf;



In [0]:
%sql

DROP TABLE IF EXISTS historical_users_bronze;

CREATE TABLE historical_users_bronze AS
SELECT *,CAST(from_unixtime(user_first_touch_timestamp/1000000) AS DATE) AS first_touch_data,
_metadata.file_modification_time AS file_modification_time,
_metadata.file_name AS file_name,
current_timestamp AS ingestion_time
FROM read_files(
    "/Volumes/workspace/default/lab_data/users-historical/",
    format => "csv",
    header => false,
    schema => "user_id STRING,
    user_first_touch_timestamp BIGINT,
    email STRING",
    rescueddatacolumn => "_rescuedata"
);

SELECT * FROM historical_users_bronze;